# Tensorflow
- TensorFlow is an open-source library developed by Google specifically for Deep Learning
- uses a dedicated data type Tensor, that can be used directly in the construction and training of neural networks. Tensors in TensorFlow are the fundamental data structure used for representing multi-dimentional arrays.
- TensorFlow has robust support for GPU acceleration, making it efficient for training depp learning models on GPUs.

Installlation 
https://www.tensorflow.org/install/pip#linux

In [16]:
import tensorflow as tf
import numpy as np

In [17]:
tf.test.is_built_with_cuda()

True

In [18]:
tf.config.list_physical_devices('GPU')

[]

# Tensor: Constant vs Variable
- tf.constant/tf.Tensor: Constants are immutable, meaining their values cannot be changed after creation. Once you create a constant tensor, its value remains fixed throughout the execution of the program. Use them for example, if you have hyperparameters or fixed values that need to remain constant throughout the trining process.
- tf.Variable: Variables are mutable, and their values can be chnaged druing the execution of the program. This makes variables suitable for situations where you need to update the values iteratively, such as in training neural network wights. Use vairables when you want to represent trainable parameters in your model, like wights and biases in a neural network.

In [19]:
x1 = tf.constant([[1.0, 2.0]]) #, dtype=np.float16)
x1

<tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[1., 2.]], dtype=float32)>

In [20]:
x1.numpy()

array([[1., 2.]], dtype=float32)

In [21]:
print(x1[0])
print(x1[0][0])

tf.Tensor([1. 2.], shape=(2,), dtype=float32)
tf.Tensor(1.0, shape=(), dtype=float32)


In [22]:
y1 = tf.convert_to_tensor(np.array([2.0, 3.0]))
y1

<tf.Tensor: shape=(2,), dtype=float64, numpy=array([2., 3.])>

In [23]:
b1 = tf.constant(4.0)
b1

<tf.Tensor: shape=(), dtype=float32, numpy=4.0>

In [24]:
b1 = tf.Variable(4.0)
b1

<tf.Variable 'Variable:0' shape=() dtype=float32, numpy=4.0>

In [25]:
# Using tf.constant
constant_tensor = tf.constant([1,2,3])

# Using tf.Variable
variable_tensor = tf.Variable([1,2,3])

# You can update the value of a tf.Variable
variable_tensor.assign([4,5,6])

# Accessing the values
print("Constant Tensor:", constant_tensor.numpy()) # output: [1 2 3]
print("Variable Tensor:", variable_tensor.numpy()) # output: [4 5 6]

Constant Tensor: [1 2 3]
Variable Tensor: [4 5 6]


## Eager vs Graph execution / Інтерактивне (послідовне) виконання проти графічного виконання

https://wwww.tensorflow.org/guide/intro_to_graphs



In [26]:
# https://www.tensorflow.org/api_docs/python/tf/executing_eagerly
# Default
assert tf.multiply(2, 3) == 6
tf.executing_eagerly()

True

In [27]:
# Simple Python function
def fn():
    print("Eager execution in a simple Python function.", tf.executing_eagerly())
    return tf.multiply(2, 3).numpy()

fn()

Eager execution in a simple Python function. True


6

In [28]:
tf.config.run_functions_eagerly(False) # Default is False, so this is just to show how to set it explicitly
@tf.function # Compiles the function into a TensorFlow graph (https://www.tensorflow.org/api_docs/python/tf/function)
def fn_graph():
    with tf.init_scope(): # Context manager, moves clause out of the graph, so it will be executed in eager mode
        print("Eager execution in a tf.function.", tf.executing_eagerly()) # Outside of the graph, so it will be executed in eager mode
    print("Eager execution inside of the graph.", tf.executing_eagerly()) # Inside of the graph, so it will be executed in graph mode
    return tf.multiply(2, 3)#.numpy()
fn_graph().numpy()

Eager execution in a tf.function. True
Eager execution inside of the graph. False


6

In [29]:
tf.config.run_functions_eagerly(True) 
@tf.function # Compiles the function into a TensorFlow graph (https://www.tensorflow.org/api_docs/python/tf/function)
def fn_eager():
    with tf.init_scope(): # Context manager, moves clause out of the graph, so it will be executed in eager mode
        print("Eager execution in a tf.function.", tf.executing_eagerly()) # Outside of the graph, so it will be executed in eager mode
    print("Eager execution inside of the graph.", tf.executing_eagerly()) # Inside of the graph, so it will be executed in graph mode
    return tf.multiply(2, 3).numpy() # This will work because we are running functions eagerly, so the graph will be executed in eager mode
fn_eager()

Eager execution in a tf.function. True
Eager execution inside of the graph. True


6

In [30]:
# fn_eager().graph()

In [31]:
# Switch to graph mode by default
# tf.compat.v1.disable_eager_execution()

In [32]:
fn_eager() # -> no eager execution anymore

Eager execution in a tf.function. True
Eager execution inside of the graph. True


6

Good practice:

* use @tf.function decorator to define your functions and switch on the option tf.config.run_functions_eagerly(True) when debugging to be able to turn it to False otherwise if needed

# Computational graph / обчислювальний граф
https://wwww.tensorflow.org/guide/intro_to_graphs

In [34]:

def formula(x: tf.Tensor, y: tf.Tensor, b: tf.Tensor) -> tf.Tensor:
    x = tf.matmul(x, y)
    x = x+b
    return x


function_that_uses_a_graph = tf.function(formula)

x1 = tf.constant([[1.0, 2.0]])
y1 = tf.constant([[2.0], [3.0]])
b1 = tf.constant([4.0])

orig_value = formula(x1, y1, b1).numpy()
tf_function_value = function_that_uses_a_graph(x1, y1, b1).numpy()

assert (orig_value == tf_function_value)
orig_value == tf_function_value

array([[ True]])

In [35]:
def my_func(x,y):
    # A  simple hand-rolled layer.
    return tf.nn.relu(tf.matmul(x, y))

my_func_tf = tf.function(my_func)

my_func_tf(x=x1, y=y1).numpy()

array([[8.]], dtype=float32)

In [37]:
# This is the graph-generating output of AutoGraph.
print(tf.autograph.to_code(my_func_tf.python_function))

def tf__my_func(x, y):
    with ag__.FunctionScope('my_func', 'fscope', ag__.ConversionOptions(recursive=True, user_requested=True, optional_features=(), internal_convert_user_code=True)) as fscope:
        do_return = False
        retval_ = ag__.UndefinedReturnValue()
        try:
            do_return = True
            retval_ = ag__.converted_call(ag__.ld(tf).nn.relu, (ag__.converted_call(ag__.ld(tf).matmul, (ag__.ld(x), ag__.ld(y)), None, fscope),), None, fscope)
        except:
            do_return = False
            raise
        return fscope.ret(retval_, do_return)



In [38]:
# This is the graph itself
print(my_func_tf.get_concrete_function(x1, y1).graph.as_graph_def())

node {
  name: "x"
  op: "Placeholder"
  attr {
    key: "shape"
    value {
      shape {
        dim {
          size: 1
        }
        dim {
          size: 2
        }
      }
    }
  }
  attr {
    key: "dtype"
    value {
      type: DT_FLOAT
    }
  }
  attr {
    key: "_user_specified_name"
    value {
      s: "x"
    }
  }
}
node {
  name: "y"
  op: "Placeholder"
  attr {
    key: "shape"
    value {
      shape {
        dim {
          size: 2
        }
        dim {
          size: 1
        }
      }
    }
  }
  attr {
    key: "dtype"
    value {
      type: DT_FLOAT
    }
  }
  attr {
    key: "_user_specified_name"
    value {
      s: "y"
    }
  }
}
node {
  name: "MatMul"
  op: "MatMul"
  input: "x"
  input: "y"
  attr {
    key: "transpose_b"
    value {
      b: false
    }
  }
  attr {
    key: "transpose_a"
    value {
      b: false
    }
  }
  attr {
    key: "grad_b"
    value {
      b: false
    }
  }
  attr {
    key: "grad_a"
    value {
      b: false

https://www.tensorflow.org/tensorboard/graphs#graphs_of_tffunctions

1. Run in terminal ```tensorboard --logdir logs/graph_only``` (NB: naviate to the notebook directory prior to running the command)
2. Go to localhost: 6006 in browser

In [ ]:
import tensorflow as tf
from datetime import datetime
import os

# 1. Create a directory 
logdir = "logs/graph_only/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(logdir)

@tf.function
def my_func(x, y):
    return tf.nn.relu(tf.matmul(x, y))

x = tf.random.uniform((3, 3))
y = tf.random.uniform((3, 3))

# 2. Use the concrete_function to extract the graph explicitly
with writer.as_default():
    # This manually pushes the graph structure to the log file
    tf.summary.graph(my_func.get_concrete_function(x, y).graph)

writer.close()
print(f"Direct graph written to: {logdir}")

Direct graph written to: logs/graph_only/20260305-204019


Швидкість виконання графа в TensorFlow порівняно з інтерактивним виконанням в основному залежить від наступних факторів:

- Оптимізація графа: Під час виконання графа TensorFlow може оптимізувати та перетворювати граф обчислень перед його виконанням. Цей процес оптимізації може включати усунення спільних підвиразів, складання констант та інші оптимізації на рівні графа.

- Розміщення пристроїв та паралельне виконання: Графічне виконання TensorFlow дозволяє ефективно розміщувати пристрої та виконувати обчислення паралельно. Граф обчислень може бути розділений на підграфи, які можуть виконуватися паралельно на кількох пристроях, таких як GPU або TPU.

- Компіляція у прискорений код: Коли використовується графічне виконання, TensorFlow має можливість компілювати граф обчислень в більш оптимізовану форму на низькому рівні, специфічні для певного обладнання.

- Зменшення накладень Python: Інтерактивне виконання включає в себе виконання операцій імперативно, що означає, що кожна операція виконується негайно в Python. Це може вводити накладення Python та обмежувати можливості оптимізації. На відміну від цього, під час графічного виконання граф будується та оптимізується перед виконанням, що зменшує необхідність у частих взаємодіях із інтерпретатором Python.

In [48]:
import timeit

x = tf.random.uniform(shape=[10, 10], minval=-1, maxval=2, dtype=tf.dtypes.int32)

def power(x, y):
    result = tf.eye(10, dtype=tf.dtypes.int32)
    for _ in range(y):
        result = tf.matmul(x, result)
    return result

print("Eager execution:", timeit.timeit(lambda: power(x, 100), number = 1000), "seconds")

Eager execution: 1.7074199790004059 seconds


In [49]:
power_as_graph = tf.function(power)
print("Graph execution:", timeit.timeit(lambda: power_as_graph(x, 100), number = 1000), "seconds")

Graph execution: 1.6917756969996844 seconds
